# Intro to Integration Testing

For this notebook, we recommend splitting the VS Code editor. You can keep this notebook open in one panel and the related files open in another while you work.

This notebook lives in `04-intro-to-integration-testing/`:

<pre>
04-intro-to-integration-testing/
├── 04-intro-to-integration-testing.ipynb
├── src/
│   └── integration_test_example/
│       ├── __init__.py
│       ├── csv_pipeline.py
│       └── db_pipeline.py
└── tests/
    ├── test_integration_csv_pipeline.py
    └── test_integration_db_pipeline.py
</pre>

An **integration test** checks that multiple components work together across real boundaries. In this notebook, those boundaries are filesystems and a [SQLite](https://www.sqlite.org/docs.html) database. Instead of validating one small function in isolation, these checks validate the handoff between reading, transforming, writing, and verifying persisted output.

That makes integration tests different from unit tests. A unit test asks whether one small piece behaves correctly by itself, while an integration test asks whether several connected pieces still behave correctly when data moves through the full workflow.

This notebook covers two examples:

* a text-file pipeline that reads, transforms, and writes rows
* a SQLite-backed pipeline that uses [SQLAlchemy](https://docs.sqlalchemy.org/en/20/) to read a table, transforms a pandas DataFrame, and writes a CSV

[pytest fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html) create the temporary files and databases used to keep both workflows isolated and repeatable.

**What to expect:** both validation modules are expected to fail until the pipeline methods in `csv_pipeline.py` and `db_pipeline.py` are implemented.

> All commands in this notebook assume you are inside `04-intro-to-integration-testing/`.

The testing pyramid is a useful scope and frequency heuristic, not a rigid quota. Narrow unit tests are usually numerous and fast, integration tests are fewer because they cross real boundaries, and end-to-end tests are kept selective because they exercise the broadest and most expensive workflows.

<div align="center">
  <img src="../assets/04-testing-pyramid.png" alt="Testing pyramid with unit, integration, and end-to-end layers" width="420" style="max-width: 100%; height: auto;" />
  <p><em>Source: <a href="https://commons.wikimedia.org/wiki/File:Testing_Pyramid.png">Testing Pyramid</a> by Mike Wacker, via Wikimedia Commons (public domain).</em></p>
</div>


In [ ]:
# Automatically reload modules when they are edited to avoid restarting the kernel.
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd().resolve(),
    Path.cwd().resolve() / "04-intro-to-integration-testing",
):
    if (candidate / "src").exists():
        NOTEBOOK_ROOT = candidate
        break
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

In [ ]:
import os

import ipytest
import pandas as pd
import pytest
from pandas.testing import assert_frame_equal
from sqlalchemy import create_engine

ipytest.autoconfig()

## Integration Pipeline Pattern

```mermaid
flowchart LR
    A["Input source"] --> B["read_data()"]
    B --> C["process_data()"]
    C --> D["write_data()"]
    C --> E["Returned result"]
    D --> F["Persisted output"]
    E --> G["Validation"]
    F --> G
```

This is the main shape to notice throughout the notebook: an integration test is not just checking one return value. It verifies that the internal steps connect correctly and that the written artifact matches the returned result.

In other words, the test is checking both orchestration and side effects. If one step is skipped, happens in the wrong order, or writes the wrong output, the integration test should make that visible.


## CSV Pipeline Walkthrough

This first example checks an end-to-end text pipeline with three steps:

1. read lines from a file
2. transform each line
3. write the transformed lines back to disk

What makes this an integration test is that it verifies orchestration, file I/O, returned values, and persisted output together.


In [ ]:
class DataPipelineCSV:
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path

    def run(self):
        data = self.read_data()
        processed_data = self.process_data(data)
        self.write_data(processed_data)
        return processed_data

    def read_data(self):
        with open(self.input_path, "r", encoding="utf-8") as file:
            return file.read().strip().split("\n")

    def process_data(self, data):
        return [item.upper() for item in data]

    def write_data(self, processed_data):
        with open(self.output_path, "w", encoding="utf-8") as file:
            file.write("\n".join(processed_data))

The fixture setup below creates temporary input and output paths so the test can run against real files without touching repository data.

This is important because integration tests should exercise realistic boundaries while still staying isolated and repeatable. Temporary files give us both: real file I/O and clean test runs.


In [ ]:
%%ipytest -qq


@pytest.fixture(scope="module")
def input_file(tmp_path_factory):
    input_path = tmp_path_factory.mktemp("data").joinpath("input.txt")
    input_path.write_text("hello\nworld\n", encoding="utf-8")
    return str(input_path)


@pytest.fixture(scope="module")
def output_file(tmp_path_factory):
    return str(tmp_path_factory.mktemp("data").joinpath("output.txt"))


def test_csv_pipeline_walkthrough(input_file, output_file):
    pipeline = DataPipelineCSV(input_file, output_file)
    processed_data = pipeline.run()

    assert os.path.exists(output_file)
    with open(output_file, "r", encoding="utf-8") as file:
        output_data = file.read().strip().split("\n")

    assert processed_data == ["HELLO", "WORLD"]
    assert output_data == ["HELLO", "WORLD"]
    assert processed_data == output_data

This walkthrough proves three things at once:

* `run()` coordinates the full flow in the right order
* the transformed result is returned to the caller
* the same transformed result is persisted to disk


## CSV Pipeline Implementation Target

Update `src/integration_test_example/csv_pipeline.py`.

1. Open `src/integration_test_example/csv_pipeline.py`.
2. Implement `run()`, `read_data()`, `process_data()`, and `write_data()`.
3. Run `../.venv/bin/python -m pytest -q tests/test_integration_csv_pipeline.py`.
4. Expect this command to fail until the stub methods are replaced, then pass once the full file-based pipeline works end to end.


In [ ]:
# @TODO Exercise 1: Build CSV integration pipeline.
# Objective: Verify read/process/write behavior end-to-end for text input.
# Edit files:
# - src/integration_test_example/csv_pipeline.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_integration_csv_pipeline.py
# Solution:
# - 04-intro-to-integration-testing.ipynb (Solution toggle block)


<details>
  <summary>Solution</summary>

```python
class DataPipelineCSV:
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path

    def run(self):
        data = self.read_data()
        processed_data = self.process_data(data)
        self.write_data(processed_data)
        return processed_data

    def read_data(self):
        with open(self.input_path, "r", encoding="utf-8") as file:
            return file.read().strip().split("\n")

    def process_data(self, data):
        return [item.upper() for item in data]

    def write_data(self, processed_data):
        with open(self.output_path, "w", encoding="utf-8") as file:
            file.write("\n".join(processed_data))
```
</details>


## Database Pipeline Walkthrough

The second example adds another boundary: a SQLite database.

This pipeline:

1. reads a table from SQLite
2. transforms a dataframe column
3. writes the processed dataframe to CSV

Again, the important idea is that the test verifies how storage, dataframe logic, and file output behave together.


In [ ]:
class DataPipelineDB:
    def __init__(self, input_path, output_path, table_name):
        self.input_path = input_path
        self.output_path = output_path
        self.table_name = table_name

    def run(self):
        data = self.read_data()
        processed_data = self.process_data(data)
        self.write_data(processed_data)
        return processed_data

    def read_data(self):
        engine = create_engine("sqlite:///" + self.input_path, echo=False)
        with engine.connect() as connection:
            return pd.read_sql_table(self.table_name, con=connection)

    def process_data(self, data):
        processed_data = data.copy()
        processed_data["names"] = processed_data["names"].str.upper()
        return processed_data

    def write_data(self, processed_data):
        processed_data.to_csv(self.output_path, index=False)

The fixtures below build a temporary SQLite database and a temporary output path. This keeps the workflow isolated while still exercising real database and filesystem boundaries.

That balance is one of the main reasons fixtures are useful in integration tests. They let the test feel realistic without depending on long-lived external state.


In [ ]:
%%ipytest -qq


@pytest.fixture(scope="module")
def input_path(tmp_path_factory):
    database_path = tmp_path_factory.mktemp("test_data").joinpath("input.db")
    engine = create_engine("sqlite:///" + str(database_path), echo=False)
    sample_data = pd.DataFrame({"names": ["mary", "john"], "age": [25, 30]})
    sample_data.to_sql("table", con=engine, index=False)
    return str(database_path)


@pytest.fixture(scope="module")
def output_path(tmp_path_factory):
    return str(tmp_path_factory.mktemp("test_data").joinpath("output.csv"))


def test_db_pipeline_walkthrough(input_path, output_path):
    pipeline = DataPipelineDB(input_path, output_path, "table")
    processed_data = pipeline.run()

    assert os.path.exists(output_path)
    output_data = pd.read_csv(output_path)

    assert_frame_equal(
        pd.DataFrame({"names": ["MARY", "JOHN"], "age": [25, 30]}),
        output_data,
    )
    assert processed_data.equals(output_data)

This walkthrough proves that the pipeline reads the expected table, transforms the correct column, returns the processed dataframe, and persists matching CSV output.

In other words, the check is validating both data correctness and workflow correctness. A failure could come from SQL access, dataframe transformation logic, or the write step itself, which is exactly why this is an integration test rather than a narrow unit test.


## Database Pipeline Implementation Target

Update `src/integration_test_example/db_pipeline.py`.

1. Open `src/integration_test_example/db_pipeline.py`.
2. Implement `run()`, `read_data()`, `process_data()`, and `write_data()`.
3. Run `../.venv/bin/python -m pytest -q tests/test_integration_db_pipeline.py`.
4. Expect this command to fail until the stub methods are replaced, then pass once the SQLite-to-CSV pipeline works end to end.


In [ ]:
# @TODO Exercise 2: Build database integration pipeline.
# Objective: Verify sqlite read, transformation, and CSV persistence in one test.
# Edit files:
# - src/integration_test_example/db_pipeline.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_integration_db_pipeline.py
# Solution:
# - 04-intro-to-integration-testing.ipynb (Solution toggle block)


<details>
  <summary>Solution</summary>

```python
import pandas as pd
from sqlalchemy import create_engine


class DataPipelineDB:
    def __init__(self, input_path, output_path, table_name):
        self.input_path = input_path
        self.output_path = output_path
        self.table_name = table_name

    def run(self):
        data = self.read_data()
        processed_data = self.process_data(data)
        self.write_data(processed_data)
        return processed_data

    def read_data(self):
        engine = create_engine("sqlite:///" + self.input_path, echo=False)
        with engine.connect() as connection:
            return pd.read_sql_table(self.table_name, con=connection)

    def process_data(self, data):
        processed_data = data.copy()
        processed_data["names"] = processed_data["names"].str.upper()
        return processed_data

    def write_data(self, processed_data):
        processed_data.to_csv(self.output_path, index=False)
```
</details>


## Running Tests

From inside `04-intro-to-integration-testing/`:

##### Validation commands:

```zsh
../.venv/bin/python -m pytest -q tests/test_integration_csv_pipeline.py
../.venv/bin/python -m pytest -q tests/test_integration_db_pipeline.py
```

**What to expect:**

* both commands should fail until the corresponding pipeline class is implemented in `src/integration_test_example/`
* the failure is expected here, because both implementation targets begin as stubs

##### Run all module tests:

```zsh
../.venv/bin/python -m pytest -q tests
```

Because this notebook intentionally includes unfinished implementation targets, the full test run will also fail until both files are completed.

There are no separate reference-solution test files in this module, so the solution blocks in this notebook are the primary reference.
